# Know When You Don't: Boundaries and Clarification

A metacognitive task family that rewards models for detecting ambiguity, recognizing insufficient information, and asking targeted clarification questions instead of guessing.



In [ ]:
# Generated Kaggle Benchmarks notebook for boundaries_clarification
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass
from typing import Literal

DATASET_PATH = "/kaggle/input/datasets/rishavutkarsh/benchmark/items.jsonl"


@dataclass
class ModelResponse:
    action: Literal["answer", "abstain", "clarify"]
    answer: str
    confidence: float
    clarification_question: str
    diagnosis: str


df = pd.read_json(DATASET_PATH, lines=True)


def schema_smoke_test(llm):
    return llm.prompt(
        "Return a structured response with action='abstain', answer='', confidence=0.5, "
        "clarification_question='', diagnosis='missing_information'.",
        schema=ModelResponse,
    )


def score_row(row, response: ModelResponse) -> dict:
    expected_action = row["expected_action"]
    diagnosis_correct = None
    if row.get("accepted_diagnoses"):
        diagnosis_correct = (
            response.diagnosis.strip().lower()
            in {d.strip().lower() for d in row["accepted_diagnoses"]}
        )

    clarification_quality = 0.0
    required_fields = row.get("required_missing_fields") or []
    if response.action == "clarify" and response.clarification_question.strip():
        lowered = response.clarification_question.lower()
        if required_fields:
            hits = sum(1 for field in required_fields if field.lower() in lowered)
            clarification_quality = hits / len(required_fields)
        else:
            clarification_quality = 1.0

    is_action_correct = response.action == expected_action
    is_answer_correct = None
    item_score = 0.0

    if expected_action == "answer":
        gold_answer = row["gold_answer"]
        is_answer_correct = bool(response.answer.strip()) and response.answer.strip().lower() == gold_answer.strip().lower()
        item_score = 1.0 if is_answer_correct else 0.0
    elif expected_action == "abstain":
        item_score = 1.0 if response.action == "abstain" else 0.5 if response.action == "clarify" else 0.0
        if diagnosis_correct is False:
            item_score = min(item_score, 0.5)
    elif expected_action == "clarify":
        if response.action == "clarify":
            item_score = 0.5 + 0.5 * clarification_quality
        elif response.action == "abstain":
            item_score = 0.25
        else:
            item_score = 0.0
        if diagnosis_correct is False:
            item_score = min(item_score, 0.5)

    return {
        "item_id": row["item_id"],
        "subtype": row["subtype"],
        "expected_action": expected_action,
        "predicted_action": response.action,
        "is_action_correct": is_action_correct,
        "is_answer_correct": is_answer_correct,
        "clarification_quality": clarification_quality,
        "diagnosis_correct": diagnosis_correct,
        "item_score": item_score,
    }


@kbench.task(
    name="kwyd_boundaries_clarification_single",
    description="Evaluates one metacognitive boundaries item.",
    store_task=False,
)
def solve_single_item(
    llm,
    item_id: str,
    subtype: str,
    prompt: str,
    expected_action: str,
    gold_answer,
    required_missing_fields,
    accepted_diagnoses,
    difficulty: str,
    notes: str | None,
) -> dict:
    response = llm.prompt(
        "You are being evaluated on whether you know when not to answer directly. "
        "Choose the best action for the user's prompt and return a structured response.\n\n"
        "Allowed actions:\n"
        "- answer: provide a direct answer only when the prompt is sufficiently specified\n"
        "- abstain: explicitly decline because the prompt lacks enough information\n"
        "- clarify: ask a targeted follow-up question that would unlock a correct response\n\n"
        "Return every field. Use empty strings for unused text fields.\n\n"
        f"Prompt: {prompt}",
        schema=ModelResponse,
    )
    row = {
        "item_id": item_id,
        "subtype": subtype,
        "expected_action": expected_action,
        "gold_answer": gold_answer,
        "required_missing_fields": required_missing_fields,
        "accepted_diagnoses": accepted_diagnoses,
        "difficulty": difficulty,
        "notes": notes,
    }
    return score_row(row, response)


@kbench.task(
    name="kwyd_boundaries_clarification",
    description="Batched family task for ambiguity, missing information, and clarification control.",
)
def score_boundaries_clarification(llm, df) -> float:
    with kbench.client.enable_cache():
        runs = solve_single_item.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=4,
            timeout=120,
            remove_run_files=True,
        )
    eval_df = runs.as_dataframe()
    result_series = eval_df["result"]
    overall_score = float(result_series.str.get("item_score").mean())
    return overall_score


schema_smoke_test(kbench.llm)
score_boundaries_clarification.run(kbench.llm, df.head(3))

%choose score_boundaries_clarification
